In [ ]:
from huggingface_hub import notebook_login
notebook_login()     # paste a Write or Read token for the Prachi1802 account, press Login

In [ ]:
import json, re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Prachi1802/ExpenseClassifier"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, device_map="auto", dtype=torch.float16).eval()

SYSTEM_PROMPT = """You check employee expenses against company policy.
Read the expense text and reply with only a JSON object with these keys:
amount, currency, category, policy_limit_amount, reimbursement_allowed, receipt_provided, flag, reason.

Categories: Travel, Ground Transport, Lodging, Meals, Software, Office Supplies, Training, Other.
Currencies: USD, EUR, GBP, INR, CAD.

Limits (USD / EUR / GBP / INR / CAD):
- Meals, per person: 75 / 70 / 60 / 6000 / 100
- Lodging, per night: 250 / 230 / 200 / 20000 / 340
- Ground Transport, per expense: 100 / 90 / 80 / 8000 / 135
- Travel, per expense: 800 / 750 / 650 / 64000 / 1100
- Software, per expense: 200 / 180 / 160 / 16000 / 270
- Office Supplies, per expense: 150 / 140 / 120 / 12000 / 200
- Training, per expense: 1000 / 900 / 800 / 80000 / 1350
- Other, per expense: 100 / 90 / 80 / 8000 / 135
A receipt is required above 25 USD, 25 EUR, 20 GBP, 2000 INR or 35 CAD.

Rules:
- policy_limit_amount = per-person limit x attendees for Meals, per-night limit x nights for Lodging, otherwise the limit. Use null if it cannot be worked out, and 0 if the expense is not reimbursable.
- receipt_provided is "No" only if the text says the receipt is missing.
- Not reimbursable: alcohol-only purchases, fines, upgrades, travel or rooms for family or friends, personal trips, hotel extras, personal subscriptions, gift cards, personal or home items, hobby classes, personal services or purchases.
- flag, first match wins: not_reimbursable; missing_info (amount, attendees, nights or a required receipt missing); over_limit (amount above policy_limit_amount); none.
- reimbursement_allowed is "No" only when the flag is not_reimbursable.
- Use null for anything the text does not state. reason briefly explains the flag."""
USER_TEMPLATE = "Expense: {text}"

def check(text):
    chat = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(text=text)}]
    inputs = tok.apply_chat_template(chat, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                             eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"),
                             pad_token_id=tok.pad_token_id)
    answer = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", answer, re.S)
    try:
        return json.loads(m.group(0))
    except Exception:
        return answer            # not valid JSON: show the raw text

# Ask as many expenses as you like; press Enter on an empty line to stop
while True:
    text = input("Expense (Enter to stop): ").strip()
    if not text:
        break
    print(json.dumps(check(text), indent=2, ensure_ascii=False), "\n")

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Expense (Enter to stop): Yesterday I spent 12000 for flight ticket
{
  "amount": 12000.0,
  "Currency": "INR",
  "category": "Travel",
  "policy_limit_amount": 64000.0,
  "Reimbursement_allowed": "Yes",
  "Receipt_provided": "Yes",
  "flag": "none",
  "Reason": "INR 12,000.00 is within the INR 64,000 Travel limit."
} 

